In [1]:
# !pip install requests pandas tqdm

import requests
import pandas as pd
import os
import time
import csv
from datetime import datetime, timezone
from urllib.parse import urlparse
from tqdm.auto import tqdm

print("✅ Dependencies loaded!")

✅ Dependencies loaded!


/Users/paramanada/Desktop/ilprl_internship/nepali_python_utils/nepali_data_tools/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# ★ EDIT THESE SETTINGS ★
# ============================================================

SUBREDDIT = "Nepal"
OUTPUT_DIR = "reddit_nepal_data"
POSTS_FILE = "post_caption.csv"
COMMENTS_FILE = "comments.csv"
MEDIA_FOLDER = "media"

POST_LIMIT = 5            # Number of text / posts to save
IMAGE_LIMIT = 5           # Number of images to download
COMMENT_LIMIT = 5         # Number of comments to save
DOWNLOAD_MEDIA = True

# ============================================================

# Create directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, MEDIA_FOLDER), exist_ok=True)

# Create session
SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json',
})

print(f"✅ Config set: r/{SUBREDDIT}")
print(f"   Output: {OUTPUT_DIR}/")
print(f"   Limits -> Posts/Text: {POST_LIMIT}, Images: {IMAGE_LIMIT}, Comments: {COMMENT_LIMIT}")


✅ Config set: r/Nepal
   Output: reddit_nepal_data/
   Limits -> Posts/Text: 5, Images: 5, Comments: 5


In [3]:
def download_image(url, post_id, index=0):
    """Download image and return filename."""
    try:
        skip = ['v.redd.it', 'youtube.com', 'youtu.be', 'gfycat.com', 'streamable', 'reddit.com/r/']
        if any(s in url.lower() for s in skip):
            return None

        ext = os.path.splitext(urlparse(url).path)[1].lower()
        if ext not in ['.jpg', '.jpeg', '.png', '.gif', '.webp']:
            if 'i.redd.it' in url or 'i.imgur.com' in url:
                ext = '.jpg'
            else:
                return None

        resp = SESSION.get(url, timeout=15, stream=True)
        if resp.status_code != 200 or 'image' not in resp.headers.get('Content-Type', ''):
            return None

        filename = f"{post_id}_{index}{ext}"
        filepath = os.path.join(OUTPUT_DIR, MEDIA_FOLDER, filename)

        with open(filepath, 'wb') as f:
            for chunk in resp.iter_content(8192):
                f.write(chunk)

        return filename
    except:
        return None


def init_csv(filepath, fieldnames):
    """Initialize CSV file with headers."""
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()


def append_csv(filepath, row, fieldnames):
    """Append single row to CSV."""
    with open(filepath, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)


def append_csv_batch(filepath, rows, fieldnames):
    """Append multiple rows to CSV."""
    with open(filepath, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerows(rows)


print("✅ Helper functions loaded!")

✅ Helper functions loaded!


In [4]:
# Post CSV columns
POST_FIELDS = ['post_id', 'title', 'selftext', 'author', 'score', 'upvote_ratio',
               'num_comments', 'created_utc', 'url', 'permalink', 'is_self',
               'flair', 'over_18', 'media_files']

# Initialize CSV
posts_path = os.path.join(OUTPUT_DIR, POSTS_FILE)
init_csv(posts_path, POST_FIELDS)

print(f"📥 Scraping r/{SUBREDDIT} posts...")
print(f"   Saving to: {posts_path}")
print("-" * 50)

# API endpoints: primary and fallback
api_endpoints = [
    "https://arctic-shift.photon-reddit.com/api/posts/search",
    "https://api.pullpush.io/reddit/search/submission/"
]

before = None
total_posts = 0
total_media = 0
POST_IDS = []

pbar = tqdm(total=POST_LIMIT or 100000, desc="Posts", unit="post")

while True:
    data = None
    for endpoint in api_endpoints:
        url = f"{endpoint}?subreddit={SUBREDDIT}&limit=50&sort=desc"
        if before:
            url += f"&before={before}"
        try:
            resp = SESSION.get(url, timeout=12)
            if resp.status_code == 200:
                res_json = resp.json()
                data = res_json.get('data', []) if isinstance(res_json, dict) else []
                if data:
                    break
        except Exception:
            continue

    if not data:
        break

    batch_rows = []
    for p in data:
        pid = p.get('id', '')
        created = p.get('created_utc', 0)
        purl = p.get('url', '')

        # Download media up to IMAGE_LIMIT
        media_files = []
        is_image_post = False
        if purl:
            exts = ['.jpg', '.jpeg', '.png', '.gif', '.webp']
            if any(e in purl.lower() for e in exts) or 'i.redd.it' in purl or 'i.imgur.com' in purl:
                is_image_post = True
                if DOWNLOAD_MEDIA and (IMAGE_LIMIT is None or total_media < IMAGE_LIMIT):
                    fname = download_image(purl, pid, 0)
                    if fname:
                        media_files.append(fname)
                        total_media += 1

        row = {
            'post_id': pid,
            'title': p.get('title', ''),
            'selftext': p.get('selftext', ''),
            'author': p.get('author', '[deleted]'),
            'score': p.get('score', 0),
            'upvote_ratio': p.get('upvote_ratio', 0),
            'num_comments': p.get('num_comments', 0),
            'created_utc': datetime.fromtimestamp(created, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S') if created else '',
            'url': purl,
            'permalink': f"https://reddit.com{p.get('permalink', '')}",
            'is_self': p.get('is_self', False),
            'flair': p.get('link_flair_text', '') or '',
            'over_18': p.get('over_18', False),
            'media_files': '|'.join(media_files)
        }

        # Record text posts up to POST_LIMIT, or include image posts if collecting images
        should_add_row = False
        if POST_LIMIT is None or total_posts < POST_LIMIT:
            total_posts += 1
            pbar.update(1)
            should_add_row = True
        elif media_files:
            should_add_row = True

        if should_add_row:
            batch_rows.append(row)
            POST_IDS.append(pid)

        # Check if both limits are fulfilled
        posts_done = POST_LIMIT is not None and total_posts >= POST_LIMIT
        images_done = (not DOWNLOAD_MEDIA) or (IMAGE_LIMIT is not None and total_media >= IMAGE_LIMIT)
        if posts_done and images_done:
            break

    if batch_rows:
        append_csv_batch(posts_path, batch_rows, POST_FIELDS)

    posts_done = POST_LIMIT is not None and total_posts >= POST_LIMIT
    images_done = (not DOWNLOAD_MEDIA) or (IMAGE_LIMIT is not None and total_media >= IMAGE_LIMIT)
    if posts_done and images_done:
        break

    before = min(p.get('created_utc', 0) for p in data)
    if before == 0:
        break

pbar.close()

print(f"\n{'='*50}")
print(f"✅ POSTS COMPLETE!")
print(f"   📄 Posts saved: {total_posts}")
print(f"   🖼️  Images saved: {total_media}")
print(f"   📁 File: {posts_path}")
print(f"{'='*50}")


📥 Scraping r/Nepal posts...
   Saving to: reddit_nepal_data/post_caption.csv
--------------------------------------------------


Posts:   0%|                                           | 0/5 [00:00<?, ?post/s]

Posts:  20%|███████                            | 1/5 [00:01<00:04,  1.12s/post]

Posts: 100%|███████████████████████████████████| 5/5 [00:02<00:00,  1.70post/s]


✅ POSTS COMPLETE!
   📄 Posts saved: 5
   🖼️  Images saved: 5
   📁 File: reddit_nepal_data/post_caption.csv


In [5]:
import shutil

output_filename = 'reddit_nepal_data'
shutil.make_archive(output_filename, 'zip', OUTPUT_DIR)
print(f"✅ Directory '{OUTPUT_DIR}' zipped to '{output_filename}.zip'")

✅ Directory 'reddit_nepal_data' zipped to 'reddit_nepal_data.zip'


In [6]:
# Comment CSV columns
COMMENT_FIELDS = ['post_id', 'post_title', 'comment_id', 'parent_id', 'author',
                  'body', 'score', 'created_utc', 'is_submitter', 'permalink']

# Initialize CSV
comments_path = os.path.join(OUTPUT_DIR, COMMENTS_FILE)
init_csv(comments_path, COMMENT_FIELDS)

print(f"📝 Scraping r/{SUBREDDIT} comments...")
print(f"   Saving to: {comments_path}")
print("-" * 50)

comment_endpoints = [
    "https://arctic-shift.photon-reddit.com/api/comments/search",
    "https://api.pullpush.io/reddit/search/comment/"
]

before = None
total_comments = 0

pbar = tqdm(total=COMMENT_LIMIT or 500000, desc="Comments", unit="comment")

while True:
    data = None
    for endpoint in comment_endpoints:
        url = f"{endpoint}?subreddit={SUBREDDIT}&limit=50&sort=desc"
        if before:
            url += f"&before={before}"
        try:
            resp = SESSION.get(url, timeout=12)
            if resp.status_code == 200:
                res_json = resp.json()
                data = res_json.get('data', []) if isinstance(res_json, dict) else []
                if data:
                    break
        except Exception:
            continue

    if not data:
        break

    batch_rows = []
    for c in data:
        created = c.get('created_utc', 0)
        row = {
            'post_id': c.get('link_id', '').replace('t3_', ''),
            'post_title': '',
            'comment_id': c.get('id', ''),
            'parent_id': c.get('parent_id', ''),
            'author': c.get('author', '[deleted]'),
            'body': c.get('body', '[deleted]'),
            'score': c.get('score', 0),
            'created_utc': datetime.fromtimestamp(created, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S') if created else '',
            'is_submitter': c.get('is_submitter', False),
            'permalink': c.get('permalink', '')
        }
        batch_rows.append(row)
        total_comments += 1
        pbar.update(1)
        if COMMENT_LIMIT and total_comments >= COMMENT_LIMIT:
            break

    append_csv_batch(comments_path, batch_rows, COMMENT_FIELDS)
    if COMMENT_LIMIT and total_comments >= COMMENT_LIMIT:
        break

    before = min(c.get('created_utc', 0) for c in data)
    if before == 0:
        break

pbar.close()

print(f"\n{'='*50}")
print(f"✅ COMMENTS COMPLETE!")
print(f"   💬 Comments saved: {total_comments}")
print(f"   📁 File: {comments_path}")
print(f"{'='*50}")


📝 Scraping r/Nepal comments...
   Saving to: reddit_nepal_data/comments.csv
--------------------------------------------------


Comments:   0%|                                     | 0/5 [00:00<?, ?comment/s]

Comments:  20%|█████▊                       | 1/5 [00:01<00:04,  1.07s/comment]

Comments: 100%|█████████████████████████████| 5/5 [00:01<00:00,  4.67comment/s]


✅ COMMENTS COMPLETE!
   💬 Comments saved: 5
   📁 File: reddit_nepal_data/comments.csv
